# SkinGen — Synthetic Skin Cancer Image Generation using CGAN

This notebook trains a **Conditional GAN (CGAN)** on the **HAM10000** dermoscopy
dataset to generate synthetic skin-lesion images conditioned on lesion class.
This helps address class imbalance for rare skin cancer subtypes.

**Pipeline:**
1. Load & preprocess HAM10000 images + metadata
2. Build a class-conditional Generator and Discriminator
3. Train the CGAN
4. Generate synthetic images per class
5. (Optional) Use synthetic images to augment a downstream classifier

> Run this notebook on **Google Colab with GPU** (Runtime → Change runtime type → GPU).


In [ ]:
# 1. Install dependencies (Colab usually has these already, listed for completeness)
!pip install -q torch torchvision pandas numpy matplotlib scikit-learn


In [ ]:
# 2. Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from sklearn.preprocessing import LabelEncoder
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Dataset

Download HAM10000 from Kaggle:
https://www.kaggle.com/datasets/kmader/skin-lesion-analysis-toward-melanoma-detection

Expected folder structure after download (update `DATA_DIR` below to match):
```
HAM10000/
├── HAM10000_metadata.csv
├── HAM10000_images_part_1/
└── HAM10000_images_part_2/
```


In [ ]:
# 4. Paths & config — EDIT DATA_DIR to point at your downloaded dataset
DATA_DIR = "/content/HAM10000"          # change if needed
METADATA_CSV = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
IMG_DIRS = [
    os.path.join(DATA_DIR, "HAM10000_images_part_1"),
    os.path.join(DATA_DIR, "HAM10000_images_part_2"),
]

IMG_SIZE = 64          # generated image resolution
BATCH_SIZE = 64
NOISE_DIM = 100
NUM_EPOCHS = 100
LR = 2e-4
BETA1 = 0.5


In [ ]:
# 5. Build a filepath lookup and label-encode the 7 lesion classes
def build_image_path_map(img_dirs):
    path_map = {}
    for d in img_dirs:
        if not os.path.isdir(d):
            continue
        for fname in os.listdir(d):
            if fname.lower().endswith(".jpg"):
                image_id = os.path.splitext(fname)[0]
                path_map[image_id] = os.path.join(d, fname)
    return path_map

metadata = pd.read_csv(METADATA_CSV)
image_path_map = build_image_path_map(IMG_DIRS)
metadata["path"] = metadata["image_id"].map(image_path_map)
metadata = metadata.dropna(subset=["path"]).reset_index(drop=True)

label_encoder = LabelEncoder()
metadata["label"] = label_encoder.fit_transform(metadata["dx"])
NUM_CLASSES = len(label_encoder.classes_)

print(f"Total usable images: {len(metadata)}")
print("Classes:", dict(zip(label_encoder.classes_, range(NUM_CLASSES))))


In [ ]:
# 6. PyTorch Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, img_size=IMG_SIZE):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),  # scale to [-1, 1]
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        label = int(row["label"])
        return img, label

dataset = HAM10000Dataset(metadata)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)


## 7. Generator and Discriminator

Both networks receive the class label as a conditioning signal via an
embedding layer, concatenated with the noise vector (Generator) or the
image (Discriminator). This is the standard CGAN formulation
(Mirza & Osindero, 2014).


In [ ]:
# 8. Generator
class Generator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM, num_classes=NUM_CLASSES, img_size=IMG_SIZE, feature_g=64):
        super().__init__()
        self.img_size = img_size
        self.label_emb = nn.Embedding(num_classes, noise_dim)

        self.net = nn.Sequential(
            # input: (noise_dim*2) x 1 x 1
            nn.ConvTranspose2d(noise_dim * 2, feature_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_g * 8), nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 8, feature_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 4), nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 4, feature_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 2), nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 2, feature_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g), nn.ReLU(True),

            nn.ConvTranspose2d(feature_g, 3, 4, 2, 1, bias=False),
            nn.Tanh(),  # output in [-1, 1], matches Normalize([0.5],[0.5])
        )

    def forward(self, noise, labels):
        label_vec = self.label_emb(labels)
        x = torch.cat([noise, label_vec], dim=1).unsqueeze(-1).unsqueeze(-1)
        return self.net(x)


In [ ]:
# 9. Discriminator
class Discriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=IMG_SIZE, feature_d=64):
        super().__init__()
        self.img_size = img_size
        self.label_emb = nn.Embedding(num_classes, img_size * img_size)

        self.net = nn.Sequential(
            # input: 4 channels (RGB + label map) x img_size x img_size
            nn.Conv2d(4, feature_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_d, feature_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 2), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_d * 2, feature_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 4), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_d * 4, feature_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 8), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, img, labels):
        label_map = self.label_emb(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, label_map], dim=1)
        return self.net(x).view(-1)


In [ ]:
# 10. Weight initialization (DCGAN-style, improves CGAN training stability)
def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator().to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))

REAL_LABEL, FAKE_LABEL = 1.0, 0.0


## 11. Training loop

In [ ]:
G_losses, D_losses = [], []

for epoch in range(NUM_EPOCHS):
    for i, (real_imgs, labels) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        b_size = real_imgs.size(0)

        # ---- Train Discriminator ----
        netD.zero_grad()
        real_targets = torch.full((b_size,), REAL_LABEL, device=device)
        output_real = netD(real_imgs, labels)
        loss_d_real = criterion(output_real, real_targets)

        noise = torch.randn(b_size, NOISE_DIM, device=device)
        fake_labels = torch.randint(0, NUM_CLASSES, (b_size,), device=device)
        fake_imgs = netG(noise, fake_labels)
        fake_targets = torch.full((b_size,), FAKE_LABEL, device=device)
        output_fake = netD(fake_imgs.detach(), fake_labels)
        loss_d_fake = criterion(output_fake, fake_targets)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        optimizerD.step()

        # ---- Train Generator ----
        netG.zero_grad()
        output = netD(fake_imgs, fake_labels)
        loss_g = criterion(output, real_targets)  # generator wants discriminator to say "real"
        loss_g.backward()
        optimizerG.step()

        if i % 50 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Batch [{i}/{len(dataloader)}] "
                  f"Loss_D: {loss_d.item():.4f} Loss_G: {loss_g.item():.4f}")

    G_losses.append(loss_g.item())
    D_losses.append(loss_d.item())

    # checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        os.makedirs("checkpoints", exist_ok=True)
        torch.save(netG.state_dict(), f"checkpoints/generator_epoch{epoch+1}.pth")


In [ ]:
# 12. Plot training losses
plt.figure(figsize=(8, 4))
plt.plot(G_losses, label="Generator")
plt.plot(D_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CGAN Training Loss")
plt.legend()
plt.show()


## 13. Generate synthetic images per class

In [ ]:
@torch.no_grad()
def generate_samples(generator, class_idx, n_samples=8):
    generator.eval()
    noise = torch.randn(n_samples, NOISE_DIM, device=device)
    labels = torch.full((n_samples,), class_idx, dtype=torch.long, device=device)
    fake_imgs = generator(noise, labels)
    fake_imgs = (fake_imgs + 1) / 2  # de-normalize from [-1,1] to [0,1]
    generator.train()
    return fake_imgs.cpu()

# Example: generate images for each class and display them
fig, axes = plt.subplots(NUM_CLASSES, 1, figsize=(8, 4 * NUM_CLASSES))
for class_idx, class_name in enumerate(label_encoder.classes_):
    samples = generate_samples(netG, class_idx, n_samples=8)
    grid = make_grid(samples, nrow=8).permute(1, 2, 0).numpy()
    ax = axes[class_idx] if NUM_CLASSES > 1 else axes
    ax.imshow(grid)
    ax.set_title(f"Synthetic samples: {class_name}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 14. Next steps

- Train a downstream classifier (e.g. ResNet) on **real + synthetic** images
  and compare accuracy/F1 on rare classes against a real-only baseline.
- Increase `IMG_SIZE` and `NUM_EPOCHS` once the pipeline is verified end-to-end.
- Track FID score between real and synthetic distributions to quantify
  generation quality.
